In [ ]:
from datasets import load_dataset
import json

# Save Data to Jsonl

In [2]:
en_dataset = load_dataset("omarkamali/wikipedia-monthly", "latest.en", split="train", streaming=True)

Resolving data files:   0%|          | 0/1425 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1432 [00:00<?, ?it/s]

In [4]:
shuffled = en_dataset.shuffle(buffer_size=10_000, seed=42)

In [ ]:

max_articles_to_process = 10_000  # Start small to test your pipeline
output_file = "rag_corpus.jsonl"

print(f"Starting extraction to {output_file}...")

with open(output_file, "w", encoding="utf-8") as f:
    for i, article in enumerate(shuffled):
        # Stop condition for testing
        if i >= max_articles_to_process:
            break
            
        # Extract relevant fields (adjust based on the dataset's specific schema)
        title = article.get('title', 'Unknown Title')
        text = article.get('text', '')
        url = article.get('url', '')
        id = article.get('id',  i)

        if not text.strip():
            continue

        # 5. Format and save each chunk with essential metadata
        record = {
            "id": id,
            "title": title,
            "url": url,
            "text": text,
        }
        # Write as JSON Lines (JSONL) - ideal for large, iterative datasets
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Successfully processed {max_articles_to_process} articles into chunks.")

Starting extraction to rag_corpus.jsonl...
Successfully processed 10000 articles into chunks.


In [7]:
id_dataset = load_dataset("omarkamali/wikipedia-monthly", "latest.id", split="train", streaming=True)
id_shuffled = id_dataset.shuffle(buffer_size=10_000, seed=42)

Resolving data files:   0%|          | 0/1425 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/153 [00:00<?, ?it/s]

In [9]:

max_articles_to_process = 100_000  # Start small to test your pipeline
output_file = "id_rag_corpus.jsonl"

print(f"Starting extraction to {output_file}...")

with open(output_file, "w", encoding="utf-8") as f:
    for i, article in enumerate(id_shuffled):
        # Stop condition for testing
        if i >= max_articles_to_process:
            break
            
        # Extract relevant fields (adjust based on the dataset's specific schema)
        title = article.get('title', 'Unknown Title')
        text = article.get('text', '')
        url = article.get('url', '')
        id = article.get('id',  i)

        if not text.strip():
            continue

        # 5. Format and save each chunk with essential metadata
        record = {
            "id": id,
            "title": title,
            "url": url,
            "text": text,
        }
        # Write as JSON Lines (JSONL) - ideal for large, iterative datasets
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Successfully processed {max_articles_to_process} articles into chunks.")

Starting extraction to id_rag_corpus.jsonl...
Successfully processed 100000 articles into chunks.


# Insert data to ChromaDB

In [13]:
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os

In [11]:
client = chromadb.PersistentClient()
collection = client.get_or_create_collection(name='file_corpus')

In [16]:
def ingest_jsonl_in_batches(filepaths, chroma_collection, batch_size):
    for file_path in filepaths:
        if not os.path.exists(file_path):
            print(f"Warning: File not found -> {file_path}")
            continue

        print(f"--- Processing {file_path} ---")
        
        documents = []
        metadatas = []
        ids = []
        inserted_count = 0
        line_count = 0

        # Open the file and process it line-by-line (streaming)
        with open(file_path, 'r', encoding='utf-8') as file:
            for line in file:
                line = line.strip()
                if not line:
                    continue # Skip empty lines
                
                line_count += 1
                
                # Parse the individual line
                try:
                    item = json.loads(line)
                except json.JSONDecodeError:
                    print(f"  -> Skipping line {line_count}: Invalid JSON format.")
                    continue

                # --- Adjust keys to match your JSONL schema ---
                text_content = item.get("text", "") 
                if not text_content:
                    continue 
                
                # Extract metadata, ensuring values are primitive types
                metadata = {k: v for k, v in item.items() if k != "text" and isinstance(v, (str, int, float, bool))}
                doc_id = str(item.get("id", f"doc_{line_count}"))

                documents.append(text_content)
                metadatas.append(metadata)
                ids.append(doc_id)

                # Insert batch when the limit is reached
                if len(documents) >= batch_size:
                    chroma_collection.add(
                        documents=documents,
                        metadatas=metadatas,
                        ids=ids
                    )
                    inserted_count += len(documents)
                    print(f"  -> Inserted {inserted_count} records so far...")
                    
                    documents.clear()
                    metadatas.clear()
                    ids.clear()

        # Catch and insert the final remainder batch after the file ends
        if documents:
            chroma_collection.add(
                documents=documents,
                metadatas=metadatas,
                ids=ids
            )
            inserted_count += len(documents)
            print(f"  -> Inserted final batch. Total: {inserted_count} records.")

        print(f"Successfully finished {file_path}.\n")

In [17]:
# Run the ingestion
corpus = ["rag_corpus.jsonl", "id_rag_corpus.jsonl"]
ingest_jsonl_in_batches(corpus, collection, batch_size=1000)

print("All files processed. Database is ready for querying.")

--- Processing rag_corpus.jsonl ---
  -> Inserted 1000 records so far...
  -> Inserted 2000 records so far...
  -> Inserted 3000 records so far...
  -> Inserted 4000 records so far...
  -> Inserted 5000 records so far...
  -> Inserted 6000 records so far...
  -> Inserted 7000 records so far...
  -> Inserted 8000 records so far...
  -> Inserted 9000 records so far...
  -> Inserted 10000 records so far...
Successfully finished rag_corpus.jsonl.

--- Processing id_rag_corpus.jsonl ---
  -> Inserted 1000 records so far...
  -> Inserted 2000 records so far...
  -> Inserted 3000 records so far...
  -> Inserted 4000 records so far...
  -> Inserted 5000 records so far...
  -> Inserted 6000 records so far...
  -> Inserted 7000 records so far...
  -> Inserted 8000 records so far...
  -> Inserted 9000 records so far...
  -> Inserted 10000 records so far...
  -> Inserted 11000 records so far...
  -> Inserted 12000 records so far...
  -> Inserted 13000 records so far...
  -> Inserted 14000 records 